# Analog Perceptron -- the math

This is the math for a single 2-input, trainable perceptron, meant to be built as real analog hardware (the rest of that project -- ESP32 training loop, power supply, level-shifting between the 3.3V digital side and the analog rails -- lives elsewhere).

**Scope, on purpose:** the analog hardware computes exactly one thing -- the weighted sum `y = w1*x1 + w2*x2 + b`. The activation function (a threshold) and the learning rule both live in software on the microcontroller side, not in analog circuitry. That keeps the physical build to the same two-op-amp summing-amplifier topology from the `01-getting-started` notebook, just with two weighted inputs instead of one, and with `w1`, `w2`, `b` meant to be driven by digital potentiometers (trainable) rather than fixed resistors.

(A hand-rolled schematic exporter lived here briefly -- dropped it in favor of pointing this `State` at an actual open-source circuit tool instead of reinventing schematic layout.)

In [1]:
from dda import State, symbols, dda, export

x1, x2, w1, w2, b, y = symbols("x1, x2, w1, w2, b, y")

## The math

Same shape as `y = m*x + b`, just with a second weighted input added into the summing junction before the inverter.

In [2]:
s = State()
s[y] = dda.neg(dda.sum(dda.mult(w1, x1), dda.mult(w2, x2), b))

s

State({'y': neg(sum(mult(w1, x1), mult(w2, x2), b))})

In [3]:
export(s, to="sympy")  # sanity check: y = w1*x1 + w2*x2 + b, no sign errors

[Eq(y, b + w1*x1 + w2*x2)]

## Next: a real circuit tool, not a hand-rolled one

Rather than growing a custom SVG schematic renderer, the plan is to point this same `State` at the [Falstad circuit simulator](https://www.falstad.com/circuit/) (open source, GPL) -- draw + live-simulate an op-amp circuit for free, in the browser, no account. Falstad has a simple line-per-component save format, so a `dda`-state -> Falstad-file exporter is a realistic next step: this math would go straight into a real, already-simulating circuit instead of a static picture.

`dda` also already has everything needed to *simulate* this same `State` numerically (`dda.scipy.to_scipy`, used for the decay/oscillator examples in `01-getting-started`) -- useful for checking the math even before the Falstad export exists.